# Practical Assignment 1 — Part III (image retrieval pipeline)
1. Best ResNet and MobileNetV3 from Part II, final classification layer removed → image embeddings.
2. STL-10 test set split into query (20%) and gallery (80%), independently within each class, fixed seed.
3. Cosine similarity between each query and all gallery images, gallery ranked by similarity, Top-K retrieval.
4. Class labels are NOT used for retrieval, only stored for evaluation.

## 1. Imports

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models
import torchvision.transforms as tt
from torchvision.datasets import STL10
from torch.utils.data import DataLoader, Subset
import numpy as np

SEED = 42
torch.manual_seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

## 2. Settings — must be the same as in Part II
Fill these in with the values used in Part II (ask your partner).

In [ ]:
RESNET_NAME    = 'resnet18'             # e.g. 'resnet18', 'resnet34', 'resnet50'
MOBILENET_NAME = 'mobilenet_v3_large'   # 'mobilenet_v3_large' or 'mobilenet_v3_small'

RESNET_PATH    = '/content/resnet_pretrained.pt'      # path to the .pt file from Part II
MOBILENET_PATH = '/content/mobilenet_pretrained.pt'   # path to the .pt file from Part II

IMG_SIZE = 224                                   # image size used in Part II
MEAN = (0.485, 0.456, 0.406)                     # normalization used in Part II
STD  = (0.229, 0.224, 0.225)

NUM_CLASSES = 10
TOP_K = 5

## 3. STL-10 test set

In [ ]:
test_transform = tt.Compose([
    tt.Resize(IMG_SIZE),
    tt.ToTensor(),
    tt.Normalize(MEAN, STD),
])

testing_data = STL10(root='./data', split='test', download=True, transform=test_transform)
classes = testing_data.classes
test_labels = np.array(testing_data.labels)
print(len(testing_data), classes)

## 4. Query / gallery split
20% query and 80% gallery, split independently within each class, fixed seed.

In [ ]:
rng = np.random.default_rng(SEED)
query_idx, gallery_idx = [], []

for c in range(NUM_CLASSES):
    class_idx = np.where(test_labels == c)[0]
    rng.shuffle(class_idx)
    n_query = int(0.2 * len(class_idx))
    query_idx.extend(class_idx[:n_query])
    gallery_idx.extend(class_idx[n_query:])

query_idx, gallery_idx = np.array(query_idx), np.array(gallery_idx)
assert len(set(query_idx) & set(gallery_idx)) == 0      # non-overlapping

print('query:', len(query_idx), '| gallery:', len(gallery_idx))
print('query images per class:', np.bincount(test_labels[query_idx]))

query_loader   = DataLoader(Subset(testing_data, query_idx),   batch_size=128, shuffle=False, num_workers=0)
gallery_loader = DataLoader(Subset(testing_data, gallery_idx), batch_size=128, shuffle=False, num_workers=0)

## 5. Load the Part II models and remove the final classification layer
After removing the last layer, the network outputs a feature vector (embedding) instead of class scores.

In [ ]:
def load_weights(model, path):
    checkpoint = torch.load(path, map_location='cpu', weights_only=False)
    if isinstance(checkpoint, nn.Module):                 # whole model was saved
        return checkpoint
    if isinstance(checkpoint, dict) and 'model_state_dict' in checkpoint:
        checkpoint = checkpoint['model_state_dict']
    model.load_state_dict(checkpoint)                     # only weights were saved
    return model

# ResNet: same architecture as in Part II, then load the trained weights
resnet = getattr(models, RESNET_NAME)(weights=None)
resnet.fc = nn.Linear(resnet.fc.in_features, NUM_CLASSES)
resnet = load_weights(resnet, RESNET_PATH)
resnet.fc = nn.Identity()                                 # remove final classification layer

# MobileNetV3: same architecture as in Part II, then load the trained weights
mobilenet = getattr(models, MOBILENET_NAME)(weights=None)
mobilenet.classifier[-1] = nn.Linear(mobilenet.classifier[-1].in_features, NUM_CLASSES)
mobilenet = load_weights(mobilenet, MOBILENET_PATH)
mobilenet.classifier[-1] = nn.Identity()                  # remove final classification layer

resnet, mobilenet = resnet.to(device).eval(), mobilenet.to(device).eval()

## 6. Extract embeddings

In [ ]:
@torch.no_grad()
def extract_embeddings(model, loader):
    embeddings, labels = [], []
    for images, y in loader:
        embeddings.append(model(images.to(device)).cpu())
        labels.append(y)                                  # stored only for evaluation
    return torch.cat(embeddings), torch.cat(labels)

embeddings = {}
for name, model in [('ResNet', resnet), ('MobileNetV3', mobilenet)]:
    q_emb, q_labels = extract_embeddings(model, query_loader)
    g_emb, g_labels = extract_embeddings(model, gallery_loader)
    embeddings[name] = {'query': q_emb, 'gallery': g_emb}
    print(f'{name}: query {tuple(q_emb.shape)}, gallery {tuple(g_emb.shape)}')

## 7. Cosine similarity, ranking and Top-K retrieval
Embeddings are L2-normalized, so the dot product equals the cosine similarity.
Only embeddings are used here — no class labels.

In [ ]:
def retrieve(query_emb, gallery_emb, k=TOP_K):
    q = F.normalize(query_emb, dim=1)
    g = F.normalize(gallery_emb, dim=1)
    similarity = q @ g.T                                          # cosine similarity, [n_query, n_gallery]
    ranking = similarity.argsort(dim=1, descending=True)          # full ranking of the gallery
    top_k = ranking[:, :k]                                        # Top-K most similar gallery images
    return similarity, ranking, top_k

results = {}
for name in embeddings:
    similarity, ranking, top_k = retrieve(embeddings[name]['query'], embeddings[name]['gallery'])
    results[name] = {'similarity': similarity, 'ranking': ranking, 'top_k': top_k}
    print(f'{name}: Top-{TOP_K} gallery positions for the first query -> {top_k[0].tolist()}')

## 8. Save everything for the evaluation part
Saved per model: query/gallery embeddings, full ranking and Top-K, plus the split indices and labels (labels only for evaluation).

In [ ]:
retrieval_output = {
    'query_idx': torch.tensor(query_idx),        # positions in the STL-10 test set
    'gallery_idx': torch.tensor(gallery_idx),
    'query_labels': q_labels,                    # for evaluation only
    'gallery_labels': g_labels,                  # for evaluation only
    'classes': classes,
}
for name in embeddings:
    retrieval_output[name] = {
        'query_emb': embeddings[name]['query'],
        'gallery_emb': embeddings[name]['gallery'],
        'ranking': results[name]['ranking'].to(torch.int16),   # int16 to keep the file small
        'top_k': results[name]['top_k'],
    }

torch.save(retrieval_output, 'retrieval_part3.pt')
print('saved retrieval_part3.pt')